In [1]:
# NOTE: Kaggle environment already includes required packages per provided list.
# Removing notebook-only pip install cells to avoid network/pip issues and keep runtime stable.

import os
import random
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torchmetrics.classification import BinaryAccuracy, BinaryAUROC

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device



device(type='cuda', index=0)

In [2]:
# Data loading utilities


class Datapreparation(object):
    def __init__(self, root_path):
        self.root_path = root_path

    def get_dataframe(self, filename):
        return pd.read_csv(os.path.join(self.root_path, filename))

    def summary(self, text, df):
        summary = pd.DataFrame(df.dtypes, columns=["dtypes"])
        summary["null"] = df.isnull().sum()
        summary["unique"] = df.nunique()
        # numeric-only stats (safe with mixed dtypes)
        num = df.select_dtypes(include=[np.number])
        if not num.empty:
            summary.loc[num.columns, "min"] = num.min()
            summary.loc[num.columns, "median"] = num.median()
            summary.loc[num.columns, "max"] = num.max()
            summary.loc[num.columns, "mean"] = num.mean()
            summary.loc[num.columns, "std"] = num.std()
        summary["duplicate"] = df.duplicated().sum()
        return summary

    def random_split_data(self, X, y):
        return train_test_split(X, y, test_size=0.20, random_state=SEED)


data = Datapreparation("/kaggle/input/playground-series-s3e18")
train_df = data.get_dataframe("train.csv")
test_df = data.get_dataframe("test.csv")

train_df.shape, test_df.shape



((13354, 38), (1484, 32))

In [3]:
# Prepare features/targets
y1 = train_df["EC1"].astype(np.float32)
y2 = train_df["EC2"].astype(np.float32)

X_df = train_df.drop(columns=["id", "EC1", "EC2", "EC3", "EC4", "EC5", "EC6"]).copy()

X_train_df, X_val_df, y1_train_s, y1_val_s = data.random_split_data(X_df, y1)
_, _, y2_train_s, y2_val_s = data.random_split_data(X_df, y2)

# IMPORTANT BUGFIX / SCORE FIX:
# Fit StandardScaler on training only, reuse for validation/test (previous code fit separately).
scaler = StandardScaler()
std_X_train = scaler.fit_transform(X_train_df)
std_X_val = scaler.transform(X_val_df)

std_X_train.shape, std_X_val.shape



((10683, 31), (2671, 31))

In [4]:
# Tensor conversion


class Tensoroperations:
    def convert_to_tensor(self, X, y=None):
        X_tensor = torch.from_numpy(np.asarray(X)).float()
        if y is None:
            return X_tensor
        y_tensor = torch.from_numpy(np.asarray(y)).float()
        return X_tensor, y_tensor


tenops = Tensoroperations()

X_tensor_train, y1_tensor_train = tenops.convert_to_tensor(
    std_X_train, y1_train_s.values
)
X_tensor_val, y1_tensor_val = tenops.convert_to_tensor(std_X_val, y1_val_s.values)

_, y2_tensor_train = tenops.convert_to_tensor(std_X_train, y2_train_s.values)
_, y2_tensor_val = tenops.convert_to_tensor(std_X_val, y2_val_s.values)

X_tensor_train.shape, y1_tensor_train.shape



(torch.Size([10683, 31]), torch.Size([10683]))

In [5]:
# Dataset definitions


class CustomDataset(Dataset):
    # BUGFIX: original code used global X_train when is_train=False, and always returned y even for test.
    def __init__(self, X_data, y_data=None, is_train=True):
        super().__init__()
        self.X_data = X_data
        self.is_train = is_train
        self.y_data = y_data if is_train else None

    def __getitem__(self, index):
        if self.is_train:
            return self.X_data[index], self.y_data[index]
        return self.X_data[index]

    def __len__(self):
        return len(self.X_data)


train1_dataset = CustomDataset(X_tensor_train, y1_tensor_train, is_train=True)
val1_dataset = CustomDataset(X_tensor_val, y1_tensor_val, is_train=True)

train2_dataset = CustomDataset(X_tensor_train, y2_tensor_train, is_train=True)
val2_dataset = CustomDataset(X_tensor_val, y2_tensor_val, is_train=True)

train_dataloader = DataLoader(train1_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val1_dataset, batch_size=64, shuffle=False)

train_dataloader2 = DataLoader(train2_dataset, batch_size=64, shuffle=True)
val_dataloader2 = DataLoader(val2_dataset, batch_size=64, shuffle=False)



In [6]:
# Model definition (kept same architecture)

false_pos, true_pos = (
    [],
    [],
)  # kept for compatibility, though ROC plotting is not required


class EnzymeClassificationBase(nn.Module):
    def _accuracy(self, outputs, labels):
        metric = BinaryAccuracy().to(outputs.device)
        return metric(outputs, labels.unsqueeze(1))

    def training_step(self, batch):
        features, labels = batch
        out = self(features)
        loss = F.binary_cross_entropy(out, labels.unsqueeze(1))
        return loss

    def validation_step(self, batch):
        features, labels = batch
        out = self(features)
        loss = F.binary_cross_entropy(out, labels.unsqueeze(1))
        acc = self._accuracy(out, labels)
        return {"Validation_loss": loss.detach(), "Validation_acc": acc.detach()}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["Validation_loss"] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        batch_accs = [x["Validation_acc"] for x in outputs]
        epoch_acc = torch.stack(batch_accs).mean()
        return {
            "Validation_loss": epoch_loss.item(),
            "Validation_acc": epoch_acc.item(),
        }

    def epoch_end(self, epoch, result):
        if epoch % 10 == 0:
            print(
                "Epoch [{}], Train_loss: {:.4f}, Validation_loss: {:.4f}, Validation_acc: {:.4f}".format(
                    epoch,
                    result["Train_loss"],
                    result["Validation_loss"],
                    result["Validation_acc"],
                )
            )


n_input_dim = std_X_train.shape[1]
n_output = 1


class MultiClassificationNN(EnzymeClassificationBase):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_input_dim, 32),
            nn.LeakyReLU(),
            nn.Linear(32, 16),
            nn.LeakyReLU(),
            nn.Linear(16, 8),
            nn.LeakyReLU(),
            nn.Linear(8, n_output),
            nn.Sigmoid(),
        )

    def forward(self, xb):
        return self.network(xb)


model_EC1 = MultiClassificationNN().to(device)
model_EC2 = MultiClassificationNN().to(device)

print(model_EC1)



MultiClassificationNN(
  (network): Sequential(
    (0): Linear(in_features=31, out_features=32, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=8, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


In [7]:
# Trainer (core loop preserved; add device move to avoid CPU/GPU mismatch)


class Trainer:
    @torch.no_grad()
    def _evaluate(self, model, val_loader):
        model.eval()
        outputs = []
        for batch in val_loader:
            features, labels = batch
            features = features.to(device)
            labels = labels.to(device)
            outputs.append(model.validation_step((features, labels)))
        return model.validation_epoch_end(outputs)

    def fit(self, epochs, lr, model, train_loader, val_loader, opt_func):
        history = []
        optimizer = opt_func(model.parameters(), lr)

        for epoch in tqdm(range(epochs), total=epochs):
            model.train()
            train_losses = []

            for batch in train_loader:
                features, labels = batch
                features = features.to(device)
                labels = labels.to(device)

                loss = model.training_step((features, labels))
                train_losses.append(loss.detach())
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

            result = self._evaluate(model, val_loader)
            result["Train_loss"] = torch.stack(train_losses).mean().item()
            model.epoch_end(epoch, result)
            history.append(result)

        return history


num_epochs = 100
lr = 1e-4
opt_func = optim.Adam

trainer = Trainer()



In [8]:
# Train EC1 and EC2

history_EC1 = trainer.fit(
    num_epochs, lr, model_EC1, train_dataloader, val_dataloader, opt_func
)
history_EC2 = trainer.fit(
    num_epochs, lr, model_EC2, train_dataloader2, val_dataloader2, opt_func
)

history_EC1[-1], history_EC2[-1]




  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [0], Train_loss: 0.6867, Validation_loss: 0.6737, Validation_acc: 0.6655
Epoch [10], Train_loss: 0.5844, Validation_loss: 0.5825, Validation_acc: 0.7026
Epoch [20], Train_loss: 0.5804, Validation_loss: 0.5790, Validation_acc: 0.7109
Epoch [30], Train_loss: 0.5787, Validation_loss: 0.5778, Validation_acc: 0.7102
Epoch [40], Train_loss: 0.5772, Validation_loss: 0.5772, Validation_acc: 0.7120
Epoch [50], Train_loss: 0.5762, Validation_loss: 0.5767, Validation_acc: 0.7109
Epoch [60], Train_loss: 0.5753, Validation_loss: 0.5765, Validation_acc: 0.7117
Epoch [70], Train_loss: 0.5743, Validation_loss: 0.5762, Validation_acc: 0.7150
Epoch [80], Train_loss: 0.5734, Validation_loss: 0.5762, Validation_acc: 0.7135
Epoch [90], Train_loss: 0.5725, Validation_loss: 0.5761, Validation_acc: 0.7135


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [0], Train_loss: 0.6881, Validation_loss: 0.6755, Validation_acc: 0.8065
Epoch [10], Train_loss: 0.5045, Validation_loss: 0.4959, Validation_acc: 0.8065
Epoch [20], Train_loss: 0.4977, Validation_loss: 0.4880, Validation_acc: 0.8065
Epoch [30], Train_loss: 0.4956, Validation_loss: 0.4860, Validation_acc: 0.8065
Epoch [40], Train_loss: 0.4945, Validation_loss: 0.4851, Validation_acc: 0.8065
Epoch [50], Train_loss: 0.4937, Validation_loss: 0.4848, Validation_acc: 0.8065
Epoch [60], Train_loss: 0.4930, Validation_loss: 0.4845, Validation_acc: 0.8065
Epoch [70], Train_loss: 0.4924, Validation_loss: 0.4845, Validation_acc: 0.8065
Epoch [80], Train_loss: 0.4918, Validation_loss: 0.4844, Validation_acc: 0.8065
Epoch [90], Train_loss: 0.4914, Validation_loss: 0.4843, Validation_acc: 0.8065


({'Validation_loss': 0.5762361288070679,
  'Validation_acc': 0.713280439376831,
  'Train_loss': 0.5718567371368408},
 {'Validation_loss': 0.4844774007797241,
  'Validation_acc': 0.8064843416213989,
  'Train_loss': 0.49106454849243164})

In [9]:
# (Optional) quick validation AUC check (not used for submission; helps sanity)
@torch.no_grad()
def compute_val_auc(model, X_tensor, y_tensor):
    model.eval()
    preds = model(X_tensor.to(device)).detach().cpu().squeeze(1)
    target = y_tensor.detach().cpu().long()
    auroc = BinaryAUROC().cpu()
    return float(auroc(preds, target))


val_auc_ec1 = compute_val_auc(model_EC1, X_tensor_val, y1_tensor_val)
val_auc_ec2 = compute_val_auc(model_EC2, X_tensor_val, y2_tensor_val)
val_auc_ec1, val_auc_ec2, (val_auc_ec1 + val_auc_ec2) / 2



(0.7037093639373779, 0.5850147008895874, 0.6443620324134827)

In [10]:
# Prepare test tensors using the SAME scaler fit on training
test_X_df = test_df.drop(columns=["id"]).copy()
std_X_test = scaler.transform(test_X_df)

X_tensor_test = tenops.convert_to_tensor(std_X_test)
test_dataset = CustomDataset(X_tensor_test, is_train=False)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

len(test_dataset), next(iter(test_dataloader)).shape




(1484, torch.Size([64, 31]))

In [11]:
# Inference
class Evaluate:
    @torch.no_grad()
    def eval_test_data(self, model, test_data_dl):
        model.eval()
        preds = []
        for X_batch_test in test_data_dl:
            X_batch_test = X_batch_test.to(device)
            y_pred = model(
                X_batch_test
            )  # already sigmoid output due to model architecture
            preds.append(y_pred.detach().cpu().squeeze(1).numpy())
        return np.concatenate(preds, axis=0)


eva = Evaluate()
ec1 = eva.eval_test_data(model_EC1, test_dataloader)
ec2 = eva.eval_test_data(model_EC2, test_dataloader)

ec1.shape, ec2.shape



((1484,), (1484,))

In [12]:
# Submission
submission = pd.DataFrame({"id": test_df["id"].values, "EC1": ec1, "EC2": ec2})
submission.to_csv("submission.csv", index=False)

submission.head(), submission.shape

(      id       EC1       EC2
 0   1377  0.758778  0.749469
 1   5485  0.671775  0.839094
 2   8753  0.430411  0.862759
 3  10746  0.724277  0.766463
 4   7554  0.478629  0.842312,
 (1484, 3))